# Tesis: Prototipo de Sistema Integrado IoT–PID–IA para Papa bajo Invernadero
## Fase 3: Dataset de Visión Artificial (Captura, Curaduría, División sin Fuga y Aumento con Albumentations)

**Ubicación:** La Paz, Bolivia (~3600 msnm, cultivo de papa bajo agrofilm térmico).
**Objetivos de esta fase:**
1. Implementar la taxonomía de 6 clases priorizadas (follaje y trampas adhesivas amarillas).
2. Integrar muestras de referencia de repositorios abiertos (**PlantVillage / PlantDoc**) con control de calidad.
3. Aplicar un pipeline de **auto-etiquetado asistido** para insectos sobre trampas cromáticas mediante segmentación HSV.
4. Diseñar un esquema de partición **70% Train, 15% Val, 15% Test agrupado por planta/unidad física**, garantizando **cero fuga de datos** (*No Data Leakage*).
5. Construir un pipeline de aumento de datos con **Albumentations** que preserve coordenadas de Bounding Boxes en formato YOLO.
6. Generar el archivo `data.yaml` estandarizado para entrenamiento con Ultralytics YOLOv8 / YOLO11 / YOLO26.

In [ ]:
# ==============================================================================
# Celda 1: Instalación de Dependencias y Configuración de Entorno
# ==============================================================================
# En Google Colab, ejecutar:
# !pip install -q albumentations opencv-python-headless pyyaml matplotlib tqdm

import os
import cv2
import yaml
import random
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import albumentations as A
from pathlib import Path

# Fijar semillas para reproducibilidad académica
random.seed(42)
np.random.seed(42)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False
print("[✔] Entorno y librerías de visión artificial inicializadas.")

In [ ]:
# ==============================================================================
# Celda 2: Montaje de Google Drive (Opcional en Colab)
# ==============================================================================
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/Tesis_IOT_Papa'
    print("[✔] Google Drive montado en /content/drive")
except Exception:
    RUTA_BASE = '.'
    print("[i] Ejecutando en entorno local.")

DATA_DIR = os.path.join(RUTA_BASE, 'data')
DATASET_YOLO = os.path.join(DATA_DIR, 'dataset_yolo')
YAML_PATH = os.path.join(DATA_DIR, 'data.yaml')

## 1. Definición de la Taxonomía de Clases

Estructuramos las 6 clases prioritarias del proyecto según la resolución óptica esperada:

In [ ]:
# ==============================================================================
# Celda 3: Diccionario de Clases y Metadatos
# ==============================================================================
CLASES = {
    0: 'tizon_tardio',    # Phytophthora infestans (lesión necrótica acuosa)
    1: 'tizon_temprano',  # Alternaria solani (lesiones concéntricas en diana)
    2: 'hoja_sana',       # Follaje vigoroso sin síntomas
    3: 'mosca_minadora',  # Liriomyza huidobrensis (adulto sobre trampa amarilla)
    4: 'pulgon',          # Myzus persicae (áfido sobre trampa amarilla)
    5: 'polilla_papa'     # Phthorimaea operculella (microlepidóptero)
}

print("Taxonomía de clases configurada:")
for k, v in CLASES.items():
    tipo = 'Trampa Amarilla (Objeto Pequeño)' if k >= 3 else 'Follaje en Dosel'
    print(f"  [{k}] {v:<16} -> {tipo}")

## 2. Auto-etiquetado Asistido para Trampas Amarillas (Visión Tradicional HSV)

Para acelerar la anotación manual de insectos diminutos (de 1 a 2 mm) sobre las láminas adhesivas amarillas, implementamos un detector asistido mediante filtrado cromático en el espacio HSV y análisis de contornos morfológicos.

In [ ]:
# ==============================================================================
# Celda 4: Función de Auto-Etiquetado Asistido por Contornos HSV
# ==============================================================================
def auto_anotar_trampa_amarilla(imagen_bgr, min_area=15, max_area=1200):
    """
    Detecta cuerpos oscuros de insectos adheridos a la lámina adhesiva amarilla.
    Retorna lista de cajas normalizadas en formato YOLO [class_id, x_center, y_center, w, h].
    """
    h_img, w_img = imagen_bgr.shape[:2]
    hsv = cv2.cvtColor(imagen_bgr, cv2.COLOR_BGR2HSV)
    
    # Máscara para fondo amarillo (Tono: 18 a 38 en OpenCV)
    bajo_amarillo = np.array([15, 60, 60])
    alto_amarillo = np.array([40, 255, 255])
    mascara_fondo = cv2.inRange(hsv, bajo_amarillo, alto_amarillo)
    
    # Los insectos son objetos no-amarillos / oscuros sobre la trampa
    mascara_insectos = cv2.bitwise_not(mascara_fondo)
    
    # Limpieza morfológica (apertura para eliminar ruido)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mascara_limpia = cv2.morphologyEx(mascara_insectos, cv2.MORPH_OPEN, kernel)
    
    contornos, _ = cv2.findContours(mascara_limpia, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    detecciones = []
    for cnt in contornos:
        area = cv2.contourArea(cnt)
        if min_area <= area <= max_area:
            x, y, w, h = cv2.boundingRect(cnt)
            # Clasificación heurística por tamaño/aspecto preliminar:
            # Área pequeña (< 80 px) -> Pulgón (ID 4)
            # Área intermedia (80 - 350 px) -> Mosca minadora (ID 3)
            # Área alargada grande (> 350 px) -> Polilla de papa (ID 5)
            if area < 80:
                c_id = 4
            elif area < 350:
                c_id = 3
            else:
                c_id = 5
                
            xc = (x + w / 2.0) / w_img
            yc = (y + h / 2.0) / h_img
            wc = w / w_img
            hc = h / h_img
            detecciones.append([c_id, xc, yc, wc, hc])
            
    return detecciones

print("[✔] Función de auto-anotación asistida cargada.")

## 3. Pipeline de Aumento de Datos con Albumentations

Configuramos un pipeline estricto de aumento adaptado a las condiciones físicas del invernadero en La Paz:
- **Invariancia Lumínica:** Variaciones de brillo, contraste y tono que simulan el agrofilm envejecido y la luz solar difusa.
- **Deformaciones Geométricas:** Rotaciones y escalados conservando la integridad de las cajas delimitadoras (*Bounding Boxes*).

In [ ]:
# ==============================================================================
# Celda 5: Definición del Pipeline de Albumentations con Parámetros YOLO
# ==============================================================================
pipeline_aumento = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), translate_percent=(-0.05, 0.05), p=0.5),
    A.MotionBlur(blur_limit=3, p=0.3),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.05, p=0.4)
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

print("[✔] Pipeline de Albumentations configurado con protección de Bounding Boxes.")

## 4. Partición 70/15/15 sin Fuga de Datos y Generación de `data.yaml`

La división se realiza **por unidad física** (`planta_id` o `trampa_id`). Las imágenes de una misma planta o trampa no se dividen entre `train` y `val`/`test`.

In [ ]:
# ==============================================================================
# Celda 6: Construcción de la Estructura YOLO y Exportación de data.yaml
# ==============================================================================
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(DATASET_YOLO, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(DATASET_YOLO, 'labels', split), exist_ok=True)

config_yaml = {
    'path': os.path.abspath(DATASET_YOLO),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASES),
    'names': CLASES
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(config_yaml, f, sort_keys=False)

print(f"[✔] data.yaml guardado en: {YAML_PATH}")
print(open(YAML_PATH).read())

## 5. Visualización de Muestras y Bounding Boxes
Comprobamos visualmente que las cajas envolventes correspondan con las clases y se mantengan consistentes tras las transformaciones de Albumentations.

In [ ]:
# ==============================================================================
# Celda 7: Visualización de Cajas Delimitadoras en Formato YOLO
# ==============================================================================
def visualizar_muestra(ruta_img, ruta_lbl):
    img = cv2.imread(ruta_img)
    if img is None: return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    if os.path.exists(ruta_lbl):
        with open(ruta_lbl, 'r') as f:
            for linea in f:
                partes = linea.strip().split()
                cid = int(partes[0])
                xc, yc, bw, bh = map(float, partes[1:5])
                x1 = int((xc - bw/2.0) * w)
                y1 = int((yc - bh/2.0) * h)
                x2 = int((xc + bw/2.0) * w)
                y2 = int((yc + bh/2.0) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(img, CLASES.get(cid, str(cid)), (x1, max(y1 - 6, 15)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img

dir_train_imgs = os.path.join(DATASET_YOLO, 'images', 'train')
dir_train_lbls = os.path.join(DATASET_YOLO, 'labels', 'train')
muestras = [f for f in os.listdir(dir_train_imgs) if f.endswith('.jpg')][:4]

if len(muestras) > 0:
    fig, axes = plt.subplots(1, min(4, len(muestras)), figsize=(14, 4), dpi=150)
    if len(muestras) == 1: axes = [axes]
    for ax, m in zip(axes, muestras):
        p_img = os.path.join(dir_train_imgs, m)
        p_lbl = os.path.join(dir_train_lbls, m.replace('.jpg', '.txt'))
        img_anotada = visualizar_muestra(p_img, p_lbl)
        ax.imshow(img_anotada)
        ax.set_title(m.split('_')[0], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("[i] No hay imágenes aún en train. Ejecuta el pipeline de preparación de la Celda anterior.")

## 6. Reporte Estadístico de Balance de Clases y Cero Fuga
Generamos la tabla de conteo de instancias para documentar el balance del dataset en la memoria escrita de tesis.

In [ ]:
# ==============================================================================
# Celda 8: Conteo y Auditoría de Distribución de Clases
# ==============================================================================
estadisticas = {s: {c: 0 for c in CLASES.values()} for s in ['train', 'val', 'test']}

for split in ['train', 'val', 'test']:
    lbl_dir = os.path.join(DATASET_YOLO, 'labels', split)
    if not os.path.exists(lbl_dir): continue
    for archivo in os.listdir(lbl_dir):
        if not archivo.endswith('.txt'): continue
        with open(os.path.join(lbl_dir, archivo), 'r') as f:
            for linea in f:
                partes = linea.strip().split()
                if partes:
                    cid = int(partes[0])
                    cname = CLASES.get(cid, 'desconocido')
                    estadisticas[split][cname] += 1

df_stats = pd.DataFrame(estadisticas)
df_stats['Total'] = df_stats.sum(axis=1)
print("=" * 60)
print("DISTRIBUCIÓN FINAL DE INSTANCIAS DEL DATASET (YOLO FORMAT)")
print("=" * 60)
print(df_stats.to_string())
print("=" * 60)
df_stats.to_csv(os.path.join(DATA_DIR, 'distribucion_clases_dataset.csv'))
print("[✔] Estadísticas exportadas a data/distribucion_clases_dataset.csv")